# nbbench thermo-seq: Sequence similarity validation

For **test vs train**, **val vs train**, and **test vs val**, compute **maximum sequence identity** per query sequence (global alignment; identity = matches / alignment length including gaps). Histograms use **count** on the y-axis; each comparison is saved as its own PNG at **350 dpi** (no figure titles).

**Dependency**: `pip install biopython` is required.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from Bio.Align import PairwiseAligner

# Reference CSVs in the same directory as this notebook
DATA_DIR = Path(".").resolve()
assert (DATA_DIR / "train.csv").exists(), f"train.csv not found in {DATA_DIR}"
print(f"Data directory: {DATA_DIR}")

Data directory: /data3/taihei/matsunaga-repos/nanobody-thermostability-sae/data/nbbench/thermo-seq


In [2]:
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_test = pd.read_csv(DATA_DIR / "test.csv")
df_val = pd.read_csv(DATA_DIR / "val.csv")

seq_train = df_train["seq"].astype(str).tolist()
seq_test = df_test["seq"].astype(str).tolist()
seq_val = df_val["seq"].astype(str).tolist()

print(f"train: {len(seq_train)} sequences")
print(f"test:  {len(seq_test)} sequences")
print(f"val:   {len(seq_val)} sequences")
lens_train = [len(s) for s in seq_train]
lens_test = [len(s) for s in seq_test]
lens_val = [len(s) for s in seq_val]
print(f"train length: min={min(lens_train)}, max={max(lens_train)}, mean={np.mean(lens_train):.1f}")
print(f"test  length: min={min(lens_test)}, max={max(lens_test)}, mean={np.mean(lens_test):.1f}")
print(f"val   length: min={min(lens_val)}, max={max(lens_val)}, mean={np.mean(lens_val):.1f}")

train: 522 sequences
test:  147 sequences
val:   95 sequences
train length: min=111, max=151, mean=123.5
test  length: min=115, max=134, mean=123.7
val   length: min=113, max=136, mean=123.8


In [3]:
# Perform global alignment of two sequences; return identity = matches / alignment length.
# Alignment length includes gaps (conventional).
def sequence_identity(seq_a: str, seq_b: str, aligner: PairwiseAligner) -> float:
    if not seq_a or not seq_b:
        return 0.0
    alns = aligner.align(seq_a, seq_b)
    aln = next(iter(alns))
    # aligned length = number of aligned pairs (including gaps)
    aln_len = aln.length
    if aln_len == 0:
        return 0.0
    matches = sum(1 for i, j in zip(aln[0], aln[1]) if i == j)
    return matches / aln_len

# Global aligner (for protein; gap penalties default)
aligner = PairwiseAligner(mode="global", substitution_matrix=None)
# BLOSUM62 etc. can be used for protein; for identity only, match/mismatch is sufficient
aligner.match_score = 1
aligner.mismatch_score = 0
aligner.open_gap_score = -1
aligner.extend_gap_score = -0.5

In [4]:
def max_identity_to_other(query_seqs: list, target_seqs: list, aligner: PairwiseAligner) -> np.ndarray:
    """For each sequence in query_seqs, return the maximum identity to any sequence in target_seqs."""
    n = len(query_seqs)
    out = np.zeros(n)
    for i, q in enumerate(query_seqs):
        best = 0.0
        for t in target_seqs:
            id_ = sequence_identity(q, t, aligner)
            if id_ > best:
                best = id_
        out[i] = best
    return out

In [5]:
print("Computing max sequence identity (test vs train, val vs train, test vs val; may take a few minutes)...")

test_vs_train = max_identity_to_other(seq_test, seq_train, aligner)
val_vs_train = max_identity_to_other(seq_val, seq_train, aligner)
test_vs_val = max_identity_to_other(seq_test, seq_val, aligner)

results = {
    "test_vs_train": test_vs_train,
    "val_vs_train": val_vs_train,
    "test_vs_val": test_vs_val,
}
print("Done.")

Computing max sequence identity (test vs train, val vs train, test vs val; may take a few minutes)...
Done.


In [13]:
bins = np.linspace(0, 1, 26)  # 0, 0.04, ..., 1.0
line_width = 0.5
label_size = 5
tick_size = 5

pairs = [
    ("test vs train", results["test_vs_train"]),
    ("val vs train", results["val_vs_train"]),
    ("test vs val", results["test_vs_val"]),
]

for label, values in pairs:
    fig, ax = plt.subplots(figsize=(2, 2))
    ax.hist(values, bins=bins, edgecolor="black", alpha=0.7, linewidth=0.5)
    ax.set_xlabel("Maximum sequence identity", fontsize=label_size)
    ax.set_ylabel("Count", fontsize=label_size)
    ax.tick_params(axis='both', width=line_width, labelsize=tick_size)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(line_width)
    ax.set_xlim(0, 1)
    plt.tight_layout()
    fname = f"sequence_similarity_{label.replace(' ', '_')}.png"
    out_path = DATA_DIR / fname
    fig.savefig(out_path, dpi=350, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out_path}")

Saved /data3/taihei/matsunaga-repos/nanobody-thermostability-sae/data/nbbench/thermo-seq/sequence_similarity_test_vs_train.png
Saved /data3/taihei/matsunaga-repos/nanobody-thermostability-sae/data/nbbench/thermo-seq/sequence_similarity_val_vs_train.png
Saved /data3/taihei/matsunaga-repos/nanobody-thermostability-sae/data/nbbench/thermo-seq/sequence_similarity_test_vs_val.png


In [ ]:
def stats(arr):
    return {"mean": np.mean(arr), "max": np.max(arr), "n_gt_80": int(np.sum(arr > 0.8))}

print("=== Summary: max sequence identity ===")
for label, key in [
    ("test vs train", "test_vs_train"),
    ("val vs train", "val_vs_train"),
    ("test vs val", "test_vs_val"),
]:
    arr = results[key]
    s = stats(arr)
    print(f"{label}: mean={s['mean']:.3f}, max={s['max']:.3f}, count > 80%: {s['n_gt_80']}")
print("\n(> 80% identity with other split may indicate potential data leakage.)")

=== Summary: max sequence identity ===
test vs train: mean=0.877, max=0.993, count > 80%: 95
val vs train: mean=0.887, max=0.992, count > 80%: 60
test vs val: mean=0.798, max=0.992, count > 80%: 40

(> 80% identity with other split may indicate potential data leakage.)
